# Lab 3.4 &mdash; Checkpointing: Resume, Approve, Rewind, Audit

**Level:** Advanced &nbsp;|&nbsp; **Est. time:** 45 min &nbsp;|&nbsp; **Day 1 &middot; Module 3 &mdash; Memory, State &amp; the LangGraph Substrate**

### What you'll do
- Attach a checkpointer and watch state survive a crash
- Stop the graph before an irreversible node with <code>interrupt_before</code>
- Resume, and rewind to an earlier checkpoint to try a different decision
- Read <code>get_state_history()</code> as the audit trail it is

> **How this lab works.** You write real LangChain and LangGraph code. Fill every `BLANK`,
> then run the **Self-check** cell under each section &mdash; those check the *objects you built*
> (a bound tool, a compiled graph, an emitted tool call), so they are deterministic and do not
> depend on the model. Cells marked **Run it for real** put your code in front of the sandbox
> model; that is the part worth watching. The score line is feedback, not a grade.

> **Builds on Lab 3.3.** Same graph. The difference is that every step is now written
> down, which is what makes approval, recovery and audit possible at all.

In [ ]:
# ---------------------------------------------------------------- Setup: run me first
import os, json, time, textwrap
from typing import Any, Callable

WORK = os.path.join("/tmp", "awmas-lab-3-04")
os.makedirs(WORK, exist_ok=True)

# ---- self-check plumbing -------------------------------------------------
_results = []

def check(name: str, fn: Callable[[], Any], hint: str = "") -> None:
    """[PASS] / [FAIL] / [TODO] for one assertion. An unfilled blank prints [TODO]."""
    try:
        ok = bool(fn())
    except NameError:
        print(f"[TODO] {name}")
        _results.append(None)
        return
    except Exception as exc:
        print(f"[FAIL] {name} -- {type(exc).__name__}: {exc}")
        _results.append(False)
        return
    print(("[PASS] " if ok else "[FAIL] ") + name + ("" if ok else (" -- " + hint if hint else "")))
    _results.append(ok)

def guard(fn: Callable[[], Any], default: Any = None) -> Any:
    """Run fn(). If a blank above is still unfilled, say so and carry on -- never crash Run All."""
    try:
        return fn()
    except NameError as exc:
        print(f"(a blank above is still unfilled: {exc} -- fill it in, then re-run this cell)")
        return default

def score() -> None:
    done = [r for r in _results if r is not None]
    passed = sum(1 for r in done if r)
    todo = sum(1 for r in _results if r is None)
    print(f"\nSelf-check: {passed}/{len(_results)}" + (f"   ({todo} still TODO)" if todo else ""))

# ---- the sandbox model ---------------------------------------------------
# Your sandbox already has an LLM configured -- nothing to install, no key to register.
# These values are read from the environment so this notebook never hardcodes an endpoint.
LLM_BASE_URL = (os.environ.get("LAB_LLM_BASE_URL") or os.environ.get("OPENAI_BASE_URL")
                or os.environ.get("LITELLM_BASE_URL"))
LLM_MODEL    = (os.environ.get("LAB_LLM_MODEL") or os.environ.get("OPENAI_MODEL")
                or os.environ.get("LITELLM_MODEL"))
LLM_API_KEY  = os.environ.get("OPENAI_API_KEY", "sandbox")

# The served model reasons before it answers, and the reasoning is billed as completion
# tokens: 24.1s / 980 tokens with it on, 0.7s / 29 with it off, for the same answer. Off is
# the default here because you will make a lot of calls today. Pass think=True to see the
# difference for yourself -- and note that prompts written as an explicit ordered procedure
# survive thinking being off, while vague ones do not.
NO_THINK = {"chat_template_kwargs": {"enable_thinking": False}}

def llm_ready() -> bool:
    if not LLM_BASE_URL or not LLM_MODEL:
        print("Model not configured. In a sandbox terminal run `env | grep -i llm` and set:")
        print("  export LAB_LLM_BASE_URL=...    # the gateway URL from your welcome sheet")
        print("  export LAB_LLM_MODEL=...       # the model name from your welcome sheet")
        return False
    return True

_llm_cache = {}
def get_llm(temperature: float = 0.0, think: bool = False):
    """A LangChain chat model pointed at the sandbox gateway (OpenAI-compatible)."""
    from langchain_openai import ChatOpenAI
    key = (temperature, think)
    if key not in _llm_cache:
        kwargs = {} if think else {"extra_body": NO_THINK}
        _llm_cache[key] = ChatOpenAI(model=LLM_MODEL, base_url=LLM_BASE_URL,
                                     api_key=LLM_API_KEY, temperature=temperature, **kwargs)
    return _llm_cache[key]

def ask(prompt: str, system: str | None = None, think: bool = False) -> str:
    """One stateless call. Returns text, or an error string -- never raises."""
    try:
        msgs = ([("system", system)] if system else []) + [("human", prompt)]
        return get_llm(think=think).invoke(msgs).content
    except Exception as exc:
        return f"<model unavailable: {type(exc).__name__}: {exc}>"

def show_messages(messages, width: int = 88) -> None:
    """Print a message list the way a trace reads: type, content, and any tool calls."""
    for m in messages:
        kind = getattr(m, "type", "?")
        body = str(getattr(m, "content", "")).replace("\n", " ")[:width]
        calls = getattr(m, "tool_calls", None)
        line = f"  [{kind:9}] {body}"
        if calls:
            line += "  -> calls: " + ", ".join(f"{c['name']}({c['args']})" for c in calls)
        print(line)

print("work dir:", WORK)
print("model   :", LLM_MODEL or "(not configured -- the object-level self-checks still work)")

In [ ]:
# ------------------------------------------------- the case file (synthetic, self-contained)
# One domain runs through all five Module 3 labs: payment exceptions on a small ledger.
# Nothing here is real data and nothing leaves this notebook.

LEDGER = {
    "PMT-1001": {"amount": 250000.00, "ccy": "USD", "counterparty": "NORTHWIND",
                 "status": "settled",  "value_date": "2026-09-01", "reason_code": None},
    "PMT-1002": {"amount":  48250.75, "ccy": "EUR", "counterparty": "ACME-EU",
                 "status": "failed",   "value_date": "2026-09-02", "reason_code": "INSUFFICIENT_FUNDS"},
    "PMT-1003": {"amount": 990000.00, "ccy": "USD", "counterparty": "ZENITH",
                 "status": "held",     "value_date": "2026-09-02", "reason_code": "LIMIT_BREACH"},
    "PMT-1004": {"amount":   1200.00, "ccy": "GBP", "counterparty": "ACME-UK",
                 "status": "failed",   "value_date": "2026-09-03", "reason_code": "INVALID_IBAN"},
    "PMT-1005": {"amount": 750000.00, "ccy": "USD", "counterparty": "NORTHWIND",
                 "status": "held",     "value_date": "2026-09-03", "reason_code": "SANCTIONS_REVIEW"},
}

POLICY = {
    "INSUFFICIENT_FUNDS": "Retry once after 24h. If it fails again, notify the client desk. No manual funding.",
    "LIMIT_BREACH":       "Payments above USD 500,000 need Treasury approval before release.",
    "INVALID_IBAN":       "Return to originator with code R04. Never repair beneficiary details in-house.",
    "SANCTIONS_REVIEW":   "Hold. Compliance decides. Operations must not release or cancel.",
}

# Which reason codes may an agent resolve on its own, and which need a human?
NEEDS_HUMAN = {"LIMIT_BREACH", "SANCTIONS_REVIEW"}

print(f"{len(LEDGER)} payments, {len(POLICY)} policy rules loaded")

In [ ]:
# ------------------------------------------------- the case tools, carried through 3.3 - 3.5
def read_ledger_record(ref: str) -> dict:
    rec = LEDGER.get(ref)
    return {"ref": ref, **rec} if rec else {"ref": ref, "error": "not_found"}

def read_policy_text(reason_code: str | None) -> str:
    return POLICY.get(reason_code, "no policy applies")

print("case helpers loaded")

In [ ]:
# ------------------------------------------------- carried forward from Lab 3.3
from typing import Annotated
from typing_extensions import TypedDict
from operator import add
from langgraph.graph import StateGraph, START, END

class CaseState(TypedDict):
    ref: str
    findings: Annotated[list, add]
    steps: int
    needs_human: bool
    answer: str | None

def read_ledger(state: CaseState) -> dict:
    rec = read_ledger_record(state["ref"])
    return {"findings": [f"ledger: status={rec.get('status')} rc={rec.get('reason_code')}"],
            "steps": state["steps"] + 1,
            "needs_human": rec.get("reason_code") in NEEDS_HUMAN}

def read_policy(state: CaseState) -> dict:
    rec = read_ledger_record(state["ref"])
    return {"findings": [f"policy: {read_policy_text(rec.get('reason_code'))}"],
            "steps": state["steps"] + 1}

def write_note(state: CaseState) -> dict:
    who = "a human must decide" if state["needs_human"] else "operations may act"
    return {"answer": f"{state['ref']}: {who}. " + " | ".join(state["findings"])}

def fresh(ref: str) -> dict:
    return {"ref": ref, "findings": [], "steps": 0, "needs_human": False, "answer": None}

print("Lab 3.3 graph pieces loaded")

## Concept

A checkpointer saves the state after **every node**, under a `thread_id`. Four capabilities fall
out of that one fact, and none of them is available without it:

| Capability | How |
|---|---|
| **resume** | re-invoke the same thread; it carries on where it stopped |
| **approve** | `interrupt_before` a node; the graph pauses, you decide, then resume |
| **rewind** | invoke from an *older* checkpoint's config and take a different branch |
| **audit** | `get_state_history()` &mdash; what was known, and when |

This is the mechanism behind human-in-the-loop, which Module 8 turns into a control.

## Section 1 &mdash; A checkpointer and a thread

`compile(checkpointer=...)` is the whole change. Everything else is the config you pass at
call time.

In [ ]:
from langgraph.checkpoint.memory import InMemorySaver

def build(checkpointer=None, interrupt_before=None):
    """The Lab 3.3 graph, now compilable with persistence and an approval gate."""
    g = StateGraph(CaseState)
    g.add_node("read_ledger", read_ledger)
    g.add_node("read_policy", read_policy)
    g.add_node("write_note", write_note)
    g.add_edge(START, "read_ledger")
    g.add_edge("read_ledger", "read_policy")
    g.add_edge("read_policy", "write_note")
    g.add_edge("write_note", END)
    return g.compile(checkpointer=checkpointer, interrupt_before=interrupt_before)


def cfg(thread_id: str) -> dict:
    return {"configurable": {"thread_id": thread_id}}

In [ ]:
# --- Self-check: Section 1   (a real checkpointed graph -- no model)
def _saved():
    saver = InMemorySaver()
    app = build(checkpointer=saver)
    app.invoke(fresh("PMT-1005"), cfg("t1"))
    return app

check("a checkpointed graph still runs",
      lambda: _saved().get_state(cfg("t1")).values["answer"] is not None)
check("the state is readable after the run",
      lambda: _saved().get_state(cfg("t1")).values["ref"] == "PMT-1005")
check("there is a checkpoint per step, not just one",
      lambda: len(list(_saved().get_state_history(cfg("t1")))) >= 4,
      "START, then one after each of the three nodes")
check("a thread that was never run is empty",
      lambda: _saved().get_state(cfg("never")).values in ({}, None),
      "threads are independent -- that is what keeps two cases apart")

## Section 2 &mdash; Pause before something irreversible

`interrupt_before=["write_note"]` stops the graph *before* that node runs and returns. The state
is saved; `get_state(...).next` tells you what it was about to do.

Resume by invoking the same thread with `None` as the input &mdash; which means "carry on", not
"start again".

In [ ]:
def start_with_gate(ref: str, thread: str, saver):
    """Run until the approval gate, then stop."""
    app = build(checkpointer=saver, interrupt_before=["write_note"])
    app.invoke(fresh(ref), cfg(thread))
    return app


def pending(app, thread: str) -> tuple:
    """What is this thread waiting to do?"""
    return app.get_state(cfg(thread)).next


def approve(app, thread: str):
    """Let it proceed. The input is None -- carry on, do not start again."""
    return app.invoke(None, cfg(thread))

In [ ]:
# --- Self-check: Section 2   (a real interrupt and resume -- no model)
def _gated():
    saver = InMemorySaver()
    app = start_with_gate("PMT-1005", "gate1", saver)
    return app

check("the graph stopped before the gated node",
      lambda: pending(_gated(), "gate1") == ("write_note",))
check("it stopped BEFORE doing the thing",
      lambda: _gated().get_state(cfg("gate1")).values["answer"] is None,
      "interrupt_before means the node has not run -- that is what makes it an approval gate")
check("the work done so far was kept",
      lambda: len(_gated().get_state(cfg("gate1")).values["findings"]) == 2,
      "a pause is not a rollback")
check("resuming finishes the run",
      lambda: approve(_gated(), "gate1")["answer"] is not None)
def _finished_has_no_next():
    saver = InMemorySaver()
    app = start_with_gate("PMT-1005", "gate2", saver)
    approve(app, "gate2")
    return app.get_state(cfg("gate2")).next == ()

check("after resuming there is nothing pending",
      lambda: _finished_has_no_next())

## Section 3 &mdash; Change your mind: update, and rewind

Two different operations, and the difference matters.

**`update_state`** writes into the *current* checkpoint &mdash; a human adding a fact before the graph
continues. It goes through the reducers, so an `Annotated[list, add]` field appends.

**Rewinding** means invoking from an *older* checkpoint's config. The graph replays from there,
and anything after it is superseded.

In [ ]:
def add_human_finding(app, thread: str, note: str):
    """A person adds something the tools could not know."""
    app.update_state(cfg(thread), {"findings": [f"human: {note}"]})
    return app.get_state(cfg(thread)).values


def checkpoint_before(app, thread: str, node: str):
    """The config of the checkpoint at which `node` was the next thing to run."""
    for snap in app.get_state_history(cfg(thread)):
        if snap.next == (node,):
            return snap.config        # a config carrying that checkpoint_id, not just the thread
    return None

In [ ]:
# --- Self-check: Section 3   (real update_state and real history -- no model)
def _updated():
    saver = InMemorySaver()
    app = start_with_gate("PMT-1005", "upd", saver)
    values = add_human_finding(app, "upd", "Compliance confirmed the hold by phone.")
    return app, values

check("the human's note went into state",
      lambda: any("human:" in f for f in _updated()[1]["findings"]))
check("update_state APPENDS rather than replacing",
      lambda: len(_updated()[1]["findings"]) == 3,
      "it goes through the reducers -- Annotated[list, add] means the tool findings survive")
check("the graph is still paused at the gate",
      lambda: _updated()[0].get_state(cfg("upd")).next == ("write_note",),
      "adding a fact is not the same as approving")
check("the resumed answer contains the human's note",
      lambda: "human:" in approve(_updated()[0], "upd")["answer"])
check("checkpoint_before finds the right point in history",
      lambda: checkpoint_before(_updated()[0], "upd", "read_policy") is not None)
check("what it returns is a checkpoint, not just the thread",
      lambda: "checkpoint_id" in checkpoint_before(_updated()[0], "upd",
                                                   "read_policy")["configurable"],
      "a config with only a thread_id points at NOW, which is not a rewind")

## Section 4 &mdash; The audit trail

`get_state_history()` returns every checkpoint, **newest first**. That is the record of what the
system knew and when it knew it &mdash; which is the question an auditor actually asks.

In [ ]:
def audit(app, thread: str) -> str:
    """The thread's history, oldest first, as something a person can read."""
    rows = []
    for snap in reversed(list(app.get_state_history(cfg(thread)))):
        rows.append(f"  next={str(snap.next):22} steps={snap.values.get('steps', 0)} "
                    f"findings={len(snap.values.get('findings', []))} "
                    f"answer={'set' if snap.values.get('answer') else '-'}")
    return "\n".join(rows)

In [ ]:
# --- Self-check: Section 4
def _history():
    saver = InMemorySaver()
    app = build(checkpointer=saver)
    app.invoke(fresh("PMT-1005"), cfg("aud"))
    return app, list(app.get_state_history(cfg("aud")))

check("history has a checkpoint per node plus the start",
      lambda: len(_history()[1]) >= 4)
check("history is returned newest first",
      lambda: _history()[1][0].values.get("answer") is not None
              and _history()[1][-1].next == ("__start__",),
      "reverse it before you show it to a person")
check("the earliest checkpoint has no findings yet",
      lambda: len(_history()[1][-1].values.get("findings", [])) == 0)
check("the audit trail is readable",
      lambda: "next=" in audit(_history()[0], "aud"))
check("every checkpoint carries its own id",
      lambda: len({s.config["configurable"]["checkpoint_id"] for s in _history()[1]})
              == len(_history()[1]),
      "identical ids would mean you cannot address a point in the past")

## Run it &mdash; crash, resume, approve, rewind

In [ ]:
def _demo():
    saver = InMemorySaver()

    print("--- 1. approval gate ---")
    app = start_with_gate("PMT-1005", "case", saver)
    print("   paused before:", pending(app, "case"))
    print("   answer so far:", app.get_state(cfg("case")).values["answer"])

    print("\n--- 2. a human adds what the tools could not know ---")
    vals = add_human_finding(app, "case", "Compliance confirmed the hold by phone at 14:02.")
    for f in vals["findings"]:
        print("   " + f[:100])

    print("\n--- 3. approve, and let it finish ---")
    out = approve(app, "case")
    print("   " + str(out["answer"])[:220])

    print("\n--- 4. rewind to before read_policy and replay ---")
    back = checkpoint_before(app, "case", "read_policy")
    if back:
        replayed = app.invoke(None, back)
        print("   replayed from an earlier checkpoint; next =",
              app.get_state(cfg("case")).next)
        print("   answer:", str(replayed.get("answer"))[:160])

    print("\n--- 5. the audit trail ---")
    print(audit(app, "case"))
guard(_demo)

### Read it

**Step 4 is the one to look at twice.** Rewinding replays the graph from an older checkpoint &mdash;
and because this graph was compiled with `interrupt_before=["write_note"]`, the replay runs
forward and then **stops at the gate again**. It does not sail through to a new answer. That is
correct, and it surprises people: the interrupt is a property of the compiled graph, not of a
particular run, so every path through it pauses. If you want the replay to finish, resume it
again.

**Step 5 is what you show an auditor.** Not the answer &mdash; the sequence. Each row is a point at
which the system had a definite set of findings and had not yet done the next thing. The question
"what did it know when it decided?" has an exact answer, and the human's note appears in it as a
finding like any other, tagged `human:` so provenance survives.

And note what made all of this available: one keyword argument. Everything in this lab is a
consequence of `compile(checkpointer=...)`. Without it, a crash loses the case, an approval gate
is impossible, and the audit question has no answer at all.

In [ ]:
score()

## Your turn

1. Swap `InMemorySaver` for `SqliteSaver` pointed at a file under `WORK`. Run the graph to the
   gate, restart the kernel, rebuild the app against the same file, and resume. That is recovery
   after a crash, and it is why in-memory is a development convenience only.
2. Gate on a **condition** rather than always: interrupt before `write_note` only when
   `needs_human` is true. (`interrupt_before` is static, so this belongs in a conditional edge to
   a node that interrupts.) Confirm PMT-1002 runs straight through.
3. Rewind, then use `update_state` to change `needs_human` to `False` before resuming. You have
   just overridden a control decision and left a record of doing so. Decide who in your
   organisation is allowed to make that call, and how the audit trail would show it.